# Price Data Imputation: MAR vs MNAR Decision

This notebook determines whether missing price data is **MAR** (Missing At Random) or **MNAR** (Missing Not At Random) and creates final imputed dataset accordingly.

## 1. Setup and Data Loading

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
# Load data
df = pd.read_excel('../data/properties_modified.xlsx')

print(f"Data loaded: {df.shape}")
print(f"Missing prices: {df['price_per_square_meter'].isna().sum():,} ({df['price_per_square_meter'].isna().mean():.1%})")

Data loaded: (8335, 10)
Missing prices: 1,151 (13.8%)


## 2. Data Preparation

In [3]:
df

,rooms,floor,material,market,price_per_square_meter,building_age,district,dist_to_metro_km,dist_to_centrum_km,river_side
0,2,0,inny,pierwotny,12990.000000,-1,Białołęka,3.507178,10.758116,wschodnia
1,2,3,inny,pierwotny,17215.754273,1,Włochy,5.217123,7.555310,zachodnia
2,1,1,beton,pierwotny,20447.154472,1,Praga Południe,4.803926,6.639387,wschodnia
3,3,4,beton,pierwotny,NaN,0,Praga Północ,1.020438,6.399839,wschodnia
4,1,2,inny,pierwotny,18033.921772,0,Praga Południe,2.284347,5.124721,wschodnia
...,...,...,...,...,...,...,...,...,...,...
8330,2,4,wielka płyta,wtórny,18161.180477,61,Ochota,2.656217,3.917136,zachodnia
8331,2,0,cegła,wtórny,13375.796178,26,Białołęka,3.633305,10.635194,wschodnia
8332,4,4,wielka płyta,wtórny,11296.076100,46,Ursynów,0.520527,7.785240,zachodnia
8333,2,4,cegła,wtórny,21694.915254,58,Wesoła,13.099249,14.940057,wschodnia


In [4]:
# Select predictors for imputation
predictor_vars = [
    'floor', 'material', 'market', 
    'building_age', 'district', 'dist_to_centrum_km', 'river_side',
    #dodane przez mnie
    'rooms', 'dist_to_metro_km'
]

# Check available variables
available_vars = [var for var in predictor_vars if var in df.columns]
print(f"Using predictors: {available_vars}")

# Prepare data for imputation
df_work = df.copy()

# Encode categorical variables
categorical_cols = df_work.select_dtypes(include=['object']).columns
le_dict = {}

for col in categorical_cols:
    if col in available_vars:
        le = LabelEncoder()
        df_work[col] = df_work[col].fillna('Unknown')
        df_work[col] = le.fit_transform(df_work[col])
        le_dict[col] = le

# Create feature matrix
X = df_work[available_vars].copy()
y = df_work['price_per_square_meter'].copy()

# Simple imputation for features (median)
feature_imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    feature_imputer.fit_transform(X), 
    columns=X.columns, 
    index=X.index
)

print(f"✅ Data prepared: {X_imputed.shape[0]:,} observations, {X_imputed.shape[1]} features")
print(f"Missing prices to impute: {y.isna().sum():,}")

Using predictors: ['floor', 'material', 'market', 'building_age', 'district', 'dist_to_centrum_km', 'river_side', 'rooms', 'dist_to_metro_km']
✅ Data prepared: 8,335 observations, 9 features
Missing prices to impute: 1,151


## 3. MICE Imputation (MAR Assumption)

In [6]:
# Create stable MICE imputation
print("🔄 Running MICE imputation...")

# Combine features and target for MICE
data_for_mice = pd.concat([X_imputed, y], axis=1)
missing_mask = y.isna()

# Set reasonable bounds based on observed prices
observed_prices = y.dropna()
lower_bound = observed_prices.quantile(0.05)  # 5th percentile
upper_bound = observed_prices.quantile(0.95)  # 95th percentile

print(f"Price bounds: {lower_bound:,.0f} to {upper_bound:,.0f}")

# Create baseline imputation
mice_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    random_state=42,
    max_iter=10,
    min_value=lower_bound,
    max_value=upper_bound
)

# Perform imputation
mice_data = mice_imputer.fit_transform(data_for_mice)
mice_df = pd.DataFrame(mice_data, columns=data_for_mice.columns, index=data_for_mice.index)

# Verify imputation
imputed_prices = mice_df.loc[missing_mask, 'price_per_square_meter']
print(f"\n✅ MICE completed:")
print(f"  Imputed {len(imputed_prices):,} prices")
print(f"  Mean: {imputed_prices.mean():,.0f}")
print(f"  Range: [{imputed_prices.min():,.0f}, {imputed_prices.max():,.0f}]")

🔄 Running MICE imputation...
Price bounds: 11,936 to 27,466

✅ MICE completed:
  Imputed 1,151 prices
  Mean: 18,327
  Range: [11,936, 24,388]


## 4. Sensitivity Analysis (MAR vs MNAR Test)

In [7]:
# Test sensitivity to MNAR assumptions
print("🔍 Testing MAR vs MNAR assumptions...")

def test_sensitivity(imputed_df, shifts=[-10, -5, 0, 5, 10]):
    """Test how model performance changes when we shift imputed values"""
    
    results = {}
    missing_mask = y.isna()
    complete_mask = y.notna()
    
    # Prepare test data (complete cases only)
    X_complete = X_imputed[complete_mask]
    y_complete = y[complete_mask]
    X_train, X_test, y_train, y_test = train_test_split(
        X_complete, y_complete, test_size=0.2, random_state=42
    )
    
    # Test each shift
    for shift_pct in shifts:
        # Create shifted dataset
        test_df = imputed_df.copy()
        
        if shift_pct != 0:
            # Shift imputed prices
            original_imputed = imputed_df.loc[missing_mask, 'price_per_square_meter']
            shifted_imputed = original_imputed * (1 + shift_pct/100)
            test_df.loc[missing_mask, 'price_per_square_meter'] = shifted_imputed
        
        # Train model with complete + imputed data
        X_full = pd.concat([X_train, test_df.loc[missing_mask, available_vars]])
        y_full = pd.concat([y_train, test_df.loc[missing_mask, 'price_per_square_meter']])
        
        # Fit model
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X_full, y_full)
        
        # Evaluate on test set (complete cases only)
        y_pred = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        results[shift_pct] = {'rmse': rmse, 'r2': r2}
    
    return results

# Run sensitivity test
sensitivity_results = test_sensitivity(mice_df)

# Display results
print("\n📊 Sensitivity Test Results:")
print("Shift% | RMSE      | R²     ")
print("-------|-----------|--------")
for shift, metrics in sensitivity_results.items():
    print(f"{shift:+4d}% | {metrics['rmse']:8.0f} | {metrics['r2']:.4f}")

# Calculate sensitivity metrics
baseline_rmse = sensitivity_results[0]['rmse']
baseline_r2 = sensitivity_results[0]['r2']

rmse_changes = [abs(results['rmse'] - baseline_rmse) / baseline_rmse 
               for shift, results in sensitivity_results.items() if shift != 0]
r2_changes = [abs(results['r2'] - baseline_r2) / abs(baseline_r2) 
             for shift, results in sensitivity_results.items() if shift != 0]

max_rmse_sensitivity = max(rmse_changes)
max_r2_sensitivity = max(r2_changes)

print(f"\n🎯 Sensitivity Metrics:")
print(f"Max RMSE change: {max_rmse_sensitivity:.1%}")
print(f"Max R² change: {max_r2_sensitivity:.1%}")

🔍 Testing MAR vs MNAR assumptions...

📊 Sensitivity Test Results:
Shift% | RMSE      | R²     
-------|-----------|--------
 -10% |     2487 | 0.7218
  -5% |     2486 | 0.7220
  +0% |     2492 | 0.7208
  +5% |     2476 | 0.7243
 +10% |     2468 | 0.7261

🎯 Sensitivity Metrics:
Max RMSE change: 1.0%
Max R² change: 0.7%


## 5. MAR vs MNAR Decision

In [8]:
# Make decision based on sensitivity analysis
print("🎯 Making MAR vs MNAR decision...")

# Decision thresholds
high_sensitivity_threshold = 0.05  # 5%
moderate_sensitivity_threshold = 0.02  # 2%

# Determine assumption
if max_rmse_sensitivity > high_sensitivity_threshold or max_r2_sensitivity > high_sensitivity_threshold:
    assumption = "MNAR"
    reason = "High sensitivity detected - imputation method affects results significantly"
    confidence = "High"
elif max_rmse_sensitivity > moderate_sensitivity_threshold or max_r2_sensitivity > moderate_sensitivity_threshold:
    assumption = "MNAR"
    reason = "Moderate sensitivity detected - some evidence of MNAR"
    confidence = "Moderate"
else:
    assumption = "MAR"
    reason = "Low sensitivity - MAR assumption appears robust"
    confidence = "High"

print(f"\n📋 DECISION SUMMARY:")
print(f"="*50)
print(f"Missing Data Assumption: {assumption}")
print(f"Confidence Level: {confidence}")
print(f"Reasoning: {reason}")
print(f"="*50)

# Store decision for final dataset creation
final_assumption = assumption

🎯 Making MAR vs MNAR decision...

📋 DECISION SUMMARY:
Missing Data Assumption: MAR
Confidence Level: High
Reasoning: Low sensitivity - MAR assumption appears robust


## 6. Create Final Imputed Dataset

In [10]:
# Create final dataset based on decision
print(f"📊 Creating final dataset based on {final_assumption} assumption...")

if final_assumption == "MAR":
    # Use standard MICE imputation
    final_dataset = mice_df.copy()
    imputation_method = "MICE (Multiple Imputation by Chained Equations)"
    
else:  # MNAR
    # For MNAR, we could implement more sophisticated methods
    # For simplicity, we'll use a conservative MICE approach with tighter bounds
    print("  Applying MNAR-aware imputation (conservative bounds)...")
    
    # Use more conservative bounds for MNAR
    conservative_lower = observed_prices.quantile(0.10)  # 10th percentile
    conservative_upper = observed_prices.quantile(0.90)  # 90th percentile
    
    mnar_imputer = IterativeImputer(
        estimator=BayesianRidge(),
        random_state=42,
        max_iter=15,  # More iterations
        min_value=conservative_lower,
        max_value=conservative_upper
    )
    
    mnar_data = mnar_imputer.fit_transform(data_for_mice)
    final_dataset = pd.DataFrame(mnar_data, columns=data_for_mice.columns, index=data_for_mice.index)
    imputation_method = "MNAR-aware MICE (conservative bounds)"

# Verify final dataset
print(f"\n✅ Final dataset created using {imputation_method}")
print(f"Dataset shape: {final_dataset.shape}")
print(f"Missing values: {final_dataset.isna().sum().sum()}")

# Show imputed price statistics
final_imputed_prices = final_dataset.loc[missing_mask, 'price_per_square_meter']
print(f"\nImputed price statistics:")
print(f"  Count: {len(final_imputed_prices):,}")
print(f"  Mean: {final_imputed_prices.mean():,.0f}")
print(f"  Std: {final_imputed_prices.std():,.0f}")
print(f"  Min: {final_imputed_prices.min():,.0f}")
print(f"  Max: {final_imputed_prices.max():,.0f}")

📊 Creating final dataset based on MAR assumption...

✅ Final dataset created using MICE (Multiple Imputation by Chained Equations)
Dataset shape: (8335, 10)
Missing values: 0

Imputed price statistics:
  Count: 1,151
  Mean: 18,327
  Std: 1,444
  Min: 11,936
  Max: 24,388


## 7. Save Final Dataset

In [11]:
# Save the final imputed dataset
output_filename = f'../data/properties_imputed_{final_assumption.lower()}.xlsx'
final_dataset.to_excel(output_filename, index=False)

print(f"💾 Final dataset saved as: {output_filename}")
print(f"\n🎯 SUMMARY:")
print(f"  • Assumption: {final_assumption}")
print(f"  • Method: {imputation_method}")
print(f"  • Imputed prices: {len(final_imputed_prices):,}")
print(f"  • File: {output_filename}")
print(f"\n✅ Ready for use in downstream analysis!")

💾 Final dataset saved as: ../data/properties_imputed_mar.xlsx

🎯 SUMMARY:
  • Assumption: MAR
  • Method: MICE (Multiple Imputation by Chained Equations)
  • Imputed prices: 1,151
  • File: ../data/properties_imputed_mar.xlsx

✅ Ready for use in downstream analysis!


In [12]:
# Quick verification - show sample of final dataset
print("📋 Sample of final dataset:")
final_dataset.head()

📋 Sample of final dataset:


,floor,material,market,building_age,district,dist_to_centrum_km,river_side,rooms,dist_to_metro_km,price_per_square_meter
0,0.0,3.0,0.0,-1.0,1.0,10.758116,0.0,2.0,3.507178,12990.000000
1,3.0,3.0,0.0,1.0,15.0,7.555310,1.0,2.0,5.217123,17215.754273
2,1.0,0.0,0.0,1.0,5.0,6.639387,0.0,1.0,4.803926,20447.154472
3,4.0,0.0,0.0,0.0,6.0,6.399839,0.0,3.0,1.020438,18019.166365
4,2.0,3.0,0.0,0.0,5.0,5.124721,0.0,1.0,2.284347,18033.921772


In [13]:
final_dataset.to_excel('../data/abt.xlsx', index=False)